# PickleSimulator: pre-simulated data with the conjugate method

`PickleSimulator` serves data that was simulated elsewhere, read from a
directory of `.pkl` files, through the same interface as the benchmark
simulators (`prior`, `simulate`, `generate_training_data`,
`generate_observed_data`).

This notebook:
1. Writes an example data directory, using the SIR simulator as a stand-in
   for an external simulator.
2. Loads it with `PickleSimulator(data_dir)`.
3. Draws training data and contaminated observed data.
4. Trains the conjugate `ConjugateModel`.
5. Calibrates $\beta$ with GPC and computes the calibrated posterior.

All settings are the values in `configs/sir.yaml`, passed directly as
function arguments.

In [1]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import torch

from nsm_bayes.inference.conjugate.calibrate_beta import calibrate_beta_gpc
from nsm_bayes.models.conjugate import ConjugateModel
from nsm_bayes.models.conjugate.conj_posterior import ConjugatePosterior
from nsm_bayes.simulators.benchmark_simulators.sir import SIRSimulator
from nsm_bayes.simulators.fromdata import PickleSimulator
from nsm_bayes.utils import set_seed

# Repository root, whether the notebook is run from examples/ or the root
REPO = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent

set_seed(123)

## 1. Writing the data directory

`PickleSimulator` expects these files in `data_dir`; each holds one array
(torch tensor, numpy array, list or DataFrame):

| file | shape | |
|---|---|---|
| `theta.pkl` | `(N, d_theta)` | parameters, required |
| `x_sim.pkl` | `(N, d_x)` | simulation at each row of `theta`, required |
| `x_obs.pkl` | `(M, d_x)` | observed data, required |
| `theta_true.pkl` | `(d_theta,)` | parameter `x_obs` was generated at, optional |
| `prior_mean.pkl`, `prior_cov.pkl` | `(d_theta,)`, `(d_theta, d_theta)` | Gaussian prior, optional |

In practice these come from your own simulator. Here we create them with
the SIR simulator, exactly as the SIR experiment does: 50,000 prior draws
with their summary statistics, and 1,000 clean observations at the true
parameter. Parameters are in unconstrained space
$(\log \beta, \log \gamma, \mathrm{logit}\,\rho, \log I_0)$.
The prior mean and true parameter in `sir.yaml` are on the natural scale
$(\beta, \gamma, \rho, I_0)$, so both are mapped to that space.

In [2]:
data_dir = REPO / "data" / "sir_pickle"
data_dir.mkdir(parents=True, exist_ok=True)

def to_unconstrained(values):
    """(beta, gamma, rho, I0) -> (log beta, log gamma, logit rho, log I0)."""
    v = torch.tensor(values, dtype=torch.float32)
    return torch.stack([v[0].log(), v[1].log(), torch.logit(v[2]), v[3].log()])

# The prior mean and true parameter are on the natural scale, so both are
# transformed; the prior standard deviations are already in unconstrained space.
sir = SIRSimulator(
    prior_mean=to_unconstrained([0.5, 0.2, 0.5, 20.0]),
    prior_cov=torch.diag(torch.tensor([0.5, 0.5, 1.0, 0.7]) ** 2),
    T=150,
    N=1000,
)
theta_true = to_unconstrained([0.6, 0.15, 0.6, 20.0])

theta, y_sim = sir.generate_training_data(50000)
y_obs = sir.simulate(theta_true.unsqueeze(0).repeat(1000, 1))

arrays = {
    "theta": theta,
    "x_sim": sir.summary(y_sim),
    "x_obs": sir.summary(y_obs),
    "theta_true": theta_true,
    "prior_mean": sir.prior_mean,
    "prior_cov": sir.prior_cov,
}
for name, arr in arrays.items():
    with open(data_dir / f"{name}.pkl", "wb") as fh:
        pickle.dump(arr, fh)
    print(f"{name}.pkl: {tuple(arr.shape)}")

ValueError: not enough values to unpack (expected 2, got 1)

## 2. Loading the data with `PickleSimulator`

The simulator takes only the data directory. The prior comes from
`prior_mean.pkl` and `prior_cov.pkl`; without them it would be a Gaussian
fitted to `theta.pkl`.

In [3]:
sim = PickleSimulator(data_dir)
prior = sim.prior

print("prior mean:", sim.prior_mean)
print("prior std: ", sim.prior_cov.diagonal().sqrt())

FileNotFoundError: Required data file not found: /u/08/sohkanj1/data/Documents/projects/nsm-bayes/data/sir_pickle/theta.pkl

## 3. Training and observed data

`generate_training_data(n)` draws `n` stored $(\theta_i, x_i)$ pairs
without replacement; with `n = 50000` that is all of them.

`generate_observed_data(theta, n_obs, epsilon)` draws `n_obs` rows of
`x_obs.pkl` and replaces a fraction $\epsilon$ of them with simulations
under random prior parameters (rows of `x_sim.pkl`), the same prior
contamination as the SIR experiment. `theta` must match `theta_true.pkl`,
because stored observations cannot be re-simulated at another parameter.

`simulate(theta)` returns the stored simulation for parameters that are
in `theta.pkl`.

In [4]:
theta_sim, x_sim = sim.generate_training_data(50000)
x_obs_mis = sim.generate_observed_data(theta_true, n_obs=100, epsilon=0.05)

assert torch.equal(sim.simulate(theta_sim[:5]), x_sim[:5])

print("theta_sim:", tuple(theta_sim.shape), " x_sim:", tuple(x_sim.shape))
print("x_obs_mis:", tuple(x_obs_mis.shape), "with 5% prior-contaminated rows")

labels_x = ["attack rate", "t_peak (scaled)", "peak (scaled)"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for j, ax in enumerate(axes):
    ax.hist(x_sim[:, j].numpy(), bins=60, density=True, alpha=0.5, label="x_sim (prior predictive)")
    ax.hist(x_obs_mis[:, j].numpy(), bins=30, density=True, alpha=0.7, label="x_obs_mis")
    ax.set_title(labels_x[j])
axes[0].legend()
plt.tight_layout()
plt.show()

NameError: name 'sim' is not defined

## 4. Training the conjugate model

`ConjugateModel` fits standardizers on the simulations and trains the
networks $T_\phi$ and $b_\phi$ by score matching, giving the surrogate
likelihood

\begin{equation*}
q_\phi(x \mid \theta) \propto \exp\{T_\phi(x)^\top \theta + b_\phi(x)\}.
\end{equation*}

In [ ]:
d_theta = theta_sim.shape[1]
model = ConjugateModel(d_x=3, d_theta=d_theta, hidden_dim=128)
model.fit(x_sim=x_sim, theta_sim=theta_sim)

history = model.train_history
plt.figure(figsize=(7, 3.5))
plt.plot(history["train_losses"], label="train")
plt.plot(history["val_losses"], label="validation")
plt.xlabel("epoch")
plt.ylabel("score-matching loss")
plt.legend()
plt.grid(True)
plt.show()

## 5. Calibrating $\beta$

The score-matching loss is not a log-likelihood, so the generalised
posterior $\pi(\theta)\,\exp\{-\beta\, n\, L(\theta; x)\}$ needs a
temperature $\beta$. Too large a $\beta$ gives an overconfident posterior;
too small a $\beta$ returns the prior. `calibrate_beta_gpc` tunes $\beta$
by general posterior calibration (GPC), matching the bootstrap coverage
of the $1 - \alpha$ credible region to its nominal level.

`robust=True` in `ConjugatePosterior` switches to robust (IMQ-weighted)
estimates of the data statistics.

In [ ]:
posterior = ConjugatePosterior(
    model,
    prior.loc,
    prior.covariance_matrix,
    robust=False,
    weight_type="imq",
)

beta, gpc_history = calibrate_beta_gpc(
    x_obs_mis,
    posterior,
    initial_beta=0.01,
    target_coverage=0.95,
    num_iterations=20,
    num_bootstraps=100,
    beta_min=1e-4,
)
print(f"calibrated beta: {beta:.4g}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(gpc_history["betas"], "o-")
ax1.set_yscale("log")
ax1.set_xlabel("GPC iteration")
ax1.set_ylabel(r"$\beta$")
ax2.plot(gpc_history["coverages"], "o-", label="bootstrap coverage")
ax2.axhline(0.95, color="k", ls="--", label="target 0.95")
ax2.set_xlabel("GPC iteration")
ax2.set_ylabel("coverage")
ax2.set_ylim(0, 1.05)
ax2.legend()
plt.tight_layout()
plt.show()

## 6. Calibrated posterior

With the calibrated $\beta$ the conjugate posterior is Gaussian in
closed form.

In [ ]:
mu_n, Sigma_n = posterior.calibrated_posterior(x_obs_mis, beta)
mu_n = mu_n.detach()
std_n = Sigma_n.detach().diagonal().sqrt()

labels = [r"$\log \beta$", r"$\log \gamma$", r"$\mathrm{logit}\,\rho$", r"$\log I_0$"]
for lab, m, s, t in zip(labels, mu_n, std_n, theta_true):
    print(f"{lab:>22}: {m:7.3f} ± {s:.3f}   (true {t:7.3f})")

fig, axes = plt.subplots(1, d_theta, figsize=(13, 3))
for i, ax in enumerate(axes):
    grid = torch.linspace(min(mu_n[i] - 4 * std_n[i], theta_true[i]),
                          max(mu_n[i] + 4 * std_n[i], theta_true[i]), 300)
    pdf = torch.distributions.Normal(mu_n[i], std_n[i]).log_prob(grid).exp()
    prior_pdf = torch.distributions.Normal(prior.loc[i], prior.covariance_matrix[i, i].sqrt()).log_prob(grid).exp()
    ax.plot(grid, pdf, label="posterior")
    ax.plot(grid, prior_pdf, color="0.6", ls=":", label="prior")
    ax.axvline(theta_true[i], color="r", ls="--", label="true")
    ax.set_title(labels[i])
    ax.set_yticks([])
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()